# Aula 2, Embeddings e vetores

Notebook executável que acompanha a aula [02-embeddings-e-vetores.md](../../lessons/modulo-09-rag/02-embeddings-e-vetores.md).

## O que vamos fazer aqui

A qualidade dos embeddings decide a qualidade da busca. Vamos ver o chunking, a quebra dos
documentos em pedaços do tamanho certo, que roda em Python puro, e deixar a recuperação densa com
Sentence Transformers como caminho opcional.

## Chunking, quebrando o documento em pedaços

Pedaços grandes diluem o assunto; pequenos perdem contexto. Uma prática comum é usar algumas
sentenças por pedaço, com sobreposição, para não cortar ideias ao meio.

In [ ]:
import re


def chunk_por_sentencas(texto, tamanho=2, sobreposicao=1):
    """Quebra o texto em pedaços de algumas sentenças, com sobreposição."""
    sentencas = [s.strip() for s in re.split(r"(?<=[.!?])\s+", texto) if s.strip()]
    pedacos = []
    passo = max(1, tamanho - sobreposicao)
    for i in range(0, len(sentencas), passo):
        pedaco = " ".join(sentencas[i:i + tamanho])
        if pedaco:
            pedacos.append(pedaco)
        if i + tamanho >= len(sentencas):
            break
    return pedacos


documento = (
    "A derivada mede a taxa de variação de uma função. "
    "Ela é a inclinação da reta tangente ao gráfico em um ponto. "
    "A regra da cadeia serve para derivar funções compostas. "
    "Já a integral acumula valores e é a operação inversa da derivada."
)

for i, p in enumerate(chunk_por_sentencas(documento, tamanho=2, sobreposicao=1)):
    print(f"Pedaço {i}: {p}")

Cada pedaço tem duas sentenças com uma de sobreposição, então ideias vizinhas não ficam
isoladas. Cada pedaço será vetorizado e indexado.

## Caminho opcional, recuperação densa com Sentence Transformers

Embeddings densos capturam sentido, não só palavras. Esta célula compara a recuperação densa com
o TF-IDF em uma pergunta com sinônimos. Roda apenas se o sentence-transformers estiver instalado.

In [ ]:
docs = [
    "A derivada mede a taxa de variação de uma função.",
    "Uma matriz organiza números em linhas e colunas.",
]
consulta = "como calcular a inclinação instantânea de uma curva"   # sinônimos de derivada

try:
    from sentence_transformers import SentenceTransformer, util
    modelo = SentenceTransformer("all-MiniLM-L6-v2")
    emb_docs = modelo.encode(docs, convert_to_tensor=True)
    emb_q = modelo.encode(consulta, convert_to_tensor=True)
    scores = util.cos_sim(emb_q, emb_docs)[0]
    melhor = int(scores.argmax())
    print("Recuperação densa escolheu:", docs[melhor])
except Exception as erro:
    print("sentence-transformers não disponível.")
    print("Com TF-IDF, a consulta com sinônimos provavelmente falharia por não compartilhar palavras.")
    print("Detalhe:", erro)

Com embeddings densos, a consulta por "inclinação instantânea de uma curva" encontra o
trecho da derivada mesmo sem repetir a palavra, algo que o TF-IDF não faria. Para o projeto, monte
um indexador com chunking configurável.